In [1]:
from wimpy import wimpy as wp
from Bio import SeqIO
import pandas as pd
import numpy as np

### Load minimap assignments

In [2]:
# promoters
PROM_IDX = pd.DataFrame(
    {"target name": ["hEf1a1", "hPGK", "CMV", "RSV"], "Assignment": [0, 1, 2, 3]}
)
THRES = 0.64

minimap_prom_df = pd.read_csv("../alignment_prom.csv")

max_idx = minimap_prom_df.groupby("name")["Perc. Align"].idxmax()
minimap_prom_assignment = minimap_prom_df.loc[max_idx].reset_index(drop=True)

# convert target name to index assignment
minimap_prom_assignment = minimap_prom_assignment.merge(
    PROM_IDX, how="left", on="target name"
).loc[:, ["name", "Perc. Align", "Assignment"]]

# apply threshold
minimap_prom_assignment.loc[
    minimap_prom_assignment["Perc. Align"] < THRES, "Assignment"
] = -1

minimap_prom_assignment.head()

,name,Perc. Align,Assignment
0,003fcd6f-f1f6-4023-993d-f3ee418ac3d8,0.942099,0
1,0074f70d-cfae-4b73-a7fb-7b3b997583ca,0.893848,0
2,00c15f1a-0a61-4198-9640-d4a4d7e59445,0.864897,0
3,00d225ef-c0c7-4b0e-ab52-d2965e87db58,0.838359,0
4,00d24c7a-3db7-48e5-88d1-52860303ae7c,0.781665,0


In [3]:
# binding sites
BS_IDX = pd.DataFrame({"target name": [2, 4, 8, 12], "Assignment": [0, 1, 2, 3]})
THRES = 0.64
minimap_bs_df = pd.read_csv("../alignment_numBS.csv")

max_idx = minimap_bs_df.groupby("name")["Perc. Align"].idxmax()
minimap_bs_assignment = minimap_bs_df.loc[max_idx].reset_index(drop=True)

# convert target name to index assignment
minimap_bs_assignment = minimap_bs_assignment.merge(
    BS_IDX, how="left", on="target name"
).loc[:, ["name", "Perc. Align", "Assignment"]]

# apply threshold
minimap_bs_assignment.loc[
    minimap_bs_assignment["Perc. Align"] < THRES, "Assignment"
] = -1
minimap_bs_assignment.head()

,name,Perc. Align,Assignment
0,003fcd6f-f1f6-4023-993d-f3ee418ac3d8,0.814208,1
1,0074f70d-cfae-4b73-a7fb-7b3b997583ca,0.644860,0
2,00c15f1a-0a61-4198-9640-d4a4d7e59445,0.626168,-1
3,00d225ef-c0c7-4b0e-ab52-d2965e87db58,0.734328,2
4,00d24c7a-3db7-48e5-88d1-52860303ae7c,0.949254,2


### Alignment with `wimpy`

In [4]:
# reference sequences from input fasta file
with open(r"../info/ref_sequences.fasta") as ref_fasta_file:
    ref_seqs = {
        record.id: str(record.seq) for record in SeqIO.parse(ref_fasta_file, "fasta")
    }
    
Puro = ref_seqs['Puro']
GFP = ref_seqs['GFP'][-100:]
A4 = ref_seqs['A4'][-50:]
mRuby = ref_seqs['mRuby'][200:300]
BS10_1 = ref_seqs['BS10_1']
promoters_100k = pd.read_excel(r'../info/100k-Promoters.xlsx')['Sequence'].to_list()


In [5]:
# load all fastq files
q_scores, lengths, seqs, names = wp.fastqall("../fastq/", return_names=True)
names = [n.split()[0] for n in names]

reading fastq files:   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
# use bowtile to re-index sequences
new_seq, _, _ = wp.bowtile(seqs, Puro, thresh=0.03)
new_seq = np.array(new_seq)

# filter out failed attempts
reads_correct = new_seq[new_seq != '']
name_reads_correct = np.array(names)[new_seq != '']
num_seqs = len(reads_correct)

bowtile progress:   0%|          | 0/3502 [00:00<?, ?it/s]

In [7]:
# aligning promoters using mRuby as landmark
_, positions_mRuby, _ = wp.tilepin_v2(reads_correct, mRuby, thresh=0.03, verbose=True)

pregions_synTF = wp.chophat(
    reads_correct,
    np.zeros_like(positions_mRuby),
    end_positions=positions_mRuby,
)

# assign promoter variants
thresh = 0.03
synTF_prom_match_ratios, _, synTF_prom_conf = wp.viscount(
    pregions_synTF, promoters_100k, thresh=thresh, tile_len=10, verbose=True
)
synTF_prom_variants = np.argmax(synTF_prom_match_ratios, axis=1)
synTF_prom_variants[np.sum(synTF_prom_match_ratios, axis=1) < thresh] = -1


match sequences to reference:   0%|          | 0/3180 [00:00<?, ?it/s]

matching to reference sequences:   0%|          | 0/4 [00:00<?, ?it/s]

In [8]:
wimpy_prom_assignment = pd.DataFrame(
    {
        "Read Name": name_reads_correct,
        "Assignment": synTF_prom_variants,
        "Perc. Align": np.max(synTF_prom_match_ratios, axis=1),
    }
)
wimpy_prom_assignment.head()

,Read Name,Assignment,Perc. Align
0,ed6336bd-7541-4a4d-83d7-299e17f6675a,1,0.667319
1,e4b61b30-e0c5-42c3-8c16-d724481e3889,2,0.986627
2,eb6c2301-79e0-4756-975c-517d4f38c3ee,2,0.945022
3,9f92c4f6-fbec-425d-b81e-b183581e055e,-1,0.000000
4,664eb670-8be0-4c79-a7fc-4f2e80f05fef,3,0.898113


In [9]:
_, positions_GFP, _ = wp.tilepin_v2(reads_correct, GFP, thresh=0.03, verbose=True)
_, positions_A4, _ = wp.tilepin_v2(reads_correct, A4, thresh=0.03, verbose=True)

p_regions = wp.chophat(
    reads_correct,
    positions=positions_A4,
    end_positions=positions_GFP,
)

nbs, _ = wp.fastar(p_regions, BS10_1, tile_len=6, bw=8)

#Assign anything with 10 or more binding sites to 12, anything between 7 & 10 binding sites to 8, and anything between 4 & 6 to 4. All else go to 0 
nbs[nbs > 9.2] = 12
nbs[(nbs > 6.9) & (nbs < 10)] = 8
nbs[(nbs > 3.9) & (nbs < 6.1)] = 4
nbs[(nbs != 2) & (nbs != 4) & (nbs != 8) & (nbs != 12)] = -1

#Reassign the number of binding sites to 0, 1, 2, 3, and 4
value_map = {-1: -1, 2: 0, 4: 1, 8: 2, 12:3}
bs_variants = [value_map[x] for x in nbs]

match sequences to reference:   0%|          | 0/3180 [00:00<?, ?it/s]

match sequences to reference:   0%|          | 0/3180 [00:00<?, ?it/s]

In [10]:
wimpy_bs_assignment = pd.DataFrame(
    {
        "Read Name": name_reads_correct,
        "Assignment": bs_variants,
        "Perc. Align": np.ones_like(nbs),
    }
)
wimpy_bs_assignment.head()

,Read Name,Assignment,Perc. Align
0,ed6336bd-7541-4a4d-83d7-299e17f6675a,3,1.0
1,e4b61b30-e0c5-42c3-8c16-d724481e3889,1,1.0
2,eb6c2301-79e0-4756-975c-517d4f38c3ee,1,1.0
3,9f92c4f6-fbec-425d-b81e-b183581e055e,-1,1.0
4,664eb670-8be0-4c79-a7fc-4f2e80f05fef,3,1.0


### Comparing results to ground truths

In [11]:
# loading ground truths for promoters
ground_truth_prom_assignment = pd.read_csv("../GroundTruth_prom_assignments.csv")
ground_truth_prom_assignment["Read Name"] = ground_truth_prom_assignment[
    "Read Name"
].apply(lambda x: x.split()[0])
assignment = ground_truth_prom_assignment["Assignment"]

# in ground truth file the promoter indices are different from what we use here
# use this mapping to convert
index_correction = {0: -1, 1: 1, 2: 2, 3: 3, 4: 0}
ground_truth_prom_assignment["Assignment"] = ground_truth_prom_assignment[
    "Assignment"
].map(index_correction)

ground_truth_prom_assignment.head()

,Read Name,Assignment
0,ed6336bd-7541-4a4d-83d7-299e17f6675a,1
1,e4b61b30-e0c5-42c3-8c16-d724481e3889,2
2,eb6c2301-79e0-4756-975c-517d4f38c3ee,2
3,9f92c4f6-fbec-425d-b81e-b183581e055e,-1
4,664eb670-8be0-4c79-a7fc-4f2e80f05fef,3


In [12]:
wimpy_performance = pd.merge(
    ground_truth_prom_assignment,
    wimpy_prom_assignment,
    on="Read Name",
    suffixes=("_True", "_WIMPY"),
    how="left",
)

accuracy = (
    wimpy_performance["Assignment_WIMPY"] == wimpy_performance["Assignment_True"]
).sum() / len(wimpy_performance)
print(f"WIMPY promoter assignment accuracy: {accuracy * 100:.2f}%")


WIMPY promoter assignment accuracy: 95.43%


In [13]:
minimap_performance = pd.merge(
    ground_truth_prom_assignment,
    minimap_prom_assignment,
    left_on="Read Name",
    right_on="name",
    suffixes=("_True", "_Minimap"),
    how="left",
)

accuracy = (
    minimap_performance["Assignment_Minimap"] == wimpy_performance["Assignment_True"]
).sum() / len(wimpy_performance)
print(f"Minimap promoter assignment accuracy: {accuracy * 100:.2f}%")


Minimap promoter assignment accuracy: 61.24%


In [14]:
# ground truths for binding sites
minP = ref_seqs["minP"]
_, positions_minP, _ = wp.tilepin_v2(reads_correct, minP, thresh=0.03, verbose=True)

bs_lengths = positions_minP - positions_A4
bs_assignment = np.digitize(bs_lengths, bins=[60, 160, 255, 415, 555]) - 1
bs_assignment[bs_assignment > 3] = -1

ground_truth_bs_assignment = pd.DataFrame(
    {
        "Read Name": name_reads_correct,
        "Assignment": bs_assignment,
    }
)
ground_truth_bs_assignment.head()

match sequences to reference:   0%|          | 0/3180 [00:00<?, ?it/s]

,Read Name,Assignment
0,ed6336bd-7541-4a4d-83d7-299e17f6675a,3
1,e4b61b30-e0c5-42c3-8c16-d724481e3889,1
2,eb6c2301-79e0-4756-975c-517d4f38c3ee,1
3,9f92c4f6-fbec-425d-b81e-b183581e055e,-1
4,664eb670-8be0-4c79-a7fc-4f2e80f05fef,3


In [15]:
wimpy_performance = pd.merge(
    ground_truth_bs_assignment,
    wimpy_bs_assignment,
    on="Read Name",
    suffixes=("_True", "_WIMPY"),
    how="left",
)

accuracy = (
    wimpy_performance["Assignment_WIMPY"] == wimpy_performance["Assignment_True"]
).sum() / len(wimpy_performance)
print(f"WIMPY binding site assignment accuracy: {accuracy * 100:.2f}%")


WIMPY binding site assignment accuracy: 95.72%


In [16]:
minimap_performance = pd.merge(
    ground_truth_bs_assignment,
    minimap_bs_assignment,
    left_on="Read Name",
    right_on="name",
    suffixes=("_True", "_Minimap"),
    how="left",
)

accuracy = (
    minimap_performance["Assignment_Minimap"] == wimpy_performance["Assignment_True"]
).sum() / len(wimpy_performance)
print(f"Minimap binding site assignment accuracy: {accuracy * 100:.2f}%")


Minimap binding site assignment accuracy: 66.23%
